### **Setup: Imports und Initialisierung**

In [8]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from quinex_application.quinex_client import QuinexClient
from quinex_application.metadata_visualisation import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [17]:
quinex = QuinexClient()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2022_abstract_fallback")

### **Create Bulk Analysis**

In [10]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quinex_application_query4 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [ ]:
visualisation.load_df(query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df
len(df_doi)

In [ ]:
visualisation.load_df(query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="22.10.2025")
df_doi = visualisation.df[
    (visualisation.df['Download_Status'] != "Succeeded")
    &
    # (visualisation.df['Download_Status'] != "Abstract Succeeded")
    # &
    (visualisation.df['Year'] >= 2010)
    ]
len(df_doi)

In [ ]:
visualisation.load_df(query='Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))', date="10.06.2025")
df_doi = visualisation.df[
    (visualisation.df['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == False)
    &
    (visualisation.df['Download_Status'] != "Succeeded")
    &
    (visualisation.df['Year'] >= 2010)
    ]
len(df_doi)

In [ ]:
df_doi

In [11]:
visualisation.load_df(query='Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))', date="12.11.2025")
df_doi = visualisation.df[
    (visualisation.df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025'] == False)
    &
    (visualisation.df['Download_Status'] != "Succeeded")
    &
    (visualisation.df['Year'] >= 2010)
    ]
len(df_doi)

3542

In [ ]:
df_1["Download_Status"].value_counts()

In [ ]:
 # Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2011 = dfs_by_year[2011]
print(len(df_2011))

In [ ]:
df_2025["Abstract"].value_counts()

In [ ]:
response = quinex.add_doi("maxime_test3", '10.1016/j.energy.2009.11.017', False)
print(response.json())
print(response.status_code)

## **Add Papers via DOI**

In [ ]:
mongodb.insert_paper(name, df_doi, force_redownload=True, only_open_access=False)

#### **Finish remaining Papers**

In [ ]:
# Hole die DOIs (_id) aus beiden Collections als Sets
papers_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2025_closed")}
failed_papers_ids = {doc["_id"] for doc in mongodb.get_collection("failed_papers")}
#papers_2025_abstract_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2025_abstract")}

# Vereine beide Sets (alle DOIs, die ausgerschlossen werden sollen)
excluded_ids = papers_ids.union(failed_papers_ids)

# Filtere das DataFrame: nur Zeilen mit DOIs, die NICHT in excluded_ids enthalten sind
filtered_df = df_2025[~df_2025["DOI"].isin(excluded_ids)]
failed_df = df_2025[df_2025["DOI"].isin(failed_papers_ids)]
print(len(filtered_df))
print(len(failed_df))

In [ ]:
mongodb.insert_paper(name, filtered_df)

## **Add Papers via Scopus CSV**

In [ ]:
df_1 = df_2011[df_2011["Abstract"].notna()]
print(len(df_1))
mongodb.insert_scopus_csv(name, df_1, force_overwrite=True)

In [ ]:
df_2 = df_2011[df_2011["Abstract"].isna()]
print(len(df_2))
mongodb.insert_paper(name, df_2, force_redownload=True, only_open_access=False, take_abstract=True)

In [ ]:
collection = mongodb.get_collection("failed_papers")
len(collection)

### **Successfully Added Papers**

In [ ]:
df_doi

In [13]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_wos_abstract_fallback", True)
print(len(dois_df))

3480


In [ ]:
dois = [doi for doi in dois_df["DOI"]]

collection = mongodb.db["papers_wos_closed_abstract"]
result = collection.delete_many({"_id": {"$in": dois}})
print(f"{result.deleted_count} Dokumente gelöscht.")

collection = mongodb.db["papers_wos_abstract"]
result = collection.delete_many({"_id": {"$in": dois}})
print(f"{result.deleted_count} Dokumente gelöscht.")

In [ ]:
collectionA = mongodb.db["failed_papers"]

collectionB = mongodb.db["papers_2020_abstract_fallback"]
ids_in_B = collectionB.distinct("_id")
result = collectionA.delete_many({"_id": {"$in": ids_in_B}})
print(f"{result.deleted_count} Dokumente gelöscht.")

# collectionB = mongodb.db["papers_wos_abstract"]
# ids_in_B = collectionB.distinct("_id")
# result = collectionA.delete_many({"_id": {"$in": ids_in_B}})
# print(f"{result.deleted_count} Dokumente gelöscht.")

## **Perform Bulk Analysis**

In [14]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=6)

🔄 Reset: 0 of 3480 requested DOIs.
✅ Analysis finished: {'detail': 'Analysis quinex_application_query4 was successfully processed.', 'processed_papers': ['W_sha256_6ecd4268dac77a905fc058ab8638ec6594211549f57db3ed265b717a32dfbdc6', 'W_sha256_4874a1d81fce62c8059c6f76a9b8d9a9906711fe2e2aec86608e51d84156276f', 'W_sha256_6347a5f526c80e8eb51a5b9cc12188d6adc777d4627899c10106eef321383c58', 'W_sha256_acd47c203e644535a043f5b0beb5defd861fc58ce87c24cad8107622b5588cff', 'W_sha256_bf59a76c01c1ab29a8eb3a239895af53e646617b1065a759e4ccab94df578a20', 'W_sha256_0472424b24c64967eeb1a901b5f0802e4b48f2274f91d05a98862fca4d255636', 'W_sha256_46c3a0f31e56b4040e6a2a7a0b0f5557798f72777f41b7f47b48949080af7af6', 'W_sha256_eeb2c817246acaed101e9e0491baa80e2750119ac6fb927f97ee3460842b7892', 'W_sha256_f12ea523d85146c151a1c7847e5d3b62989171a89a1bf798f7ec59c1b6371998', 'W_sha256_d2c8641296d97778a3743c6f52f9555e478b9d04adb6e8605a9926203cda1906', 'W_sha256_6a34da6d583a84173c6f65e253fb0871dd45ed87b423f16ac3bd59247cc8a4fc',

## **Add Bulk Analysis to MongoDB**

In [15]:
# wos (!) und noch machen
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)


📊 Successfully processed: 3480 of 3480 DOIs


## **Update Paper Parquet**

In [16]:
df = mongodb.update_paper_parquet("papers_wos_abstract_fallback", "../data/paper/paper_dataset.parquet", "24.11.2025", "17.04.2025", abstract=True)

In Collection papers_wos_abstract_fallback -> Quinex True: 2250, Quinex False: 1230


In [ ]:
import pandas as pd
df = pd.read_parquet("../data/paper/paper_dataset.parquet")

In [ ]:
df[df["Year"] == 2014]

In [ ]:
print("-------------------QUINEX Performed-------------------------------")
print(len(df[(df["Quinex"] == True)]))
print("-------------------QUINEX Tried-----------------------------------")
print(len(df[(df["Download_Status"] == "Succeeded")]) +  len(df[(df["Download_Status"] == "Abstract Succeeded")]) + len(df[(df["Download_Status"] == "Failed")]))
print("-------------------Overall Papers-------------------------------")
print(len(df[
    (
        (df['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True)
        | 
        (df['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == True)
        | 
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)
        |
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025'] == True)
    )
    & 
    (df['Year'] >= 2010)
]))

In [ ]:
print(len(df[(df["Download_Status"] == "Abstract Succeeded")]))
print(len(df[(df["Download_Status"] == "Succeeded")]))
print(len(df[(df["Download_Status"] == "Failed")]))